In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.ensemble import RandomForestRegressor



In [4]:
import pandas as pd
file_path = "/content/drive/MyDrive/value-train.csv"
df = pd.read_csv(file_path)

In [5]:
target_col = "target"
y = df[target_col]
x = df.drop(columns=[target_col, "ID"])

cat_cols = x.select_dtypes(include=["object"]).columns.to_list()
num_cols = x.select_dtypes(include=["int64", "float64"]).columns.to_list()


In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (3567, 4991)
X_valid: (892, 4991)
y_train: (3567,)
y_valid: (892,)


In [7]:
variance_selector = VarianceThreshold(threshold=0.01)

X_train_var = variance_selector.fit_transform(X_train)
X_valid_var = variance_selector.transform(X_valid)

print(X_train_var.shape)
print(X_valid_var.shape)

(3567, 4669)
(892, 4669)


In [18]:
X_train_var = pd.DataFrame(
    X_train_var,
    columns=X_train.columns[variance_selector.get_support()]
)

X_valid_var = pd.DataFrame(
    X_valid_var,
    columns=X_train.columns[variance_selector.get_support()]
)

In [19]:
class CorrelationFilter(BaseEstimator, TransformerMixin):

    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()

        upper = corr_matrix.where(
            np.triu(
                np.ones(corr_matrix.shape),
                k=1
            ).astype(bool)
        )

        self.features_to_drop_ = [
            column
            for column in upper.columns
            if any(upper[column] > self.threshold)
        ]

        self.features_to_keep_ = [
            column
            for column in X.columns
            if column not in self.features_to_drop_
        ]

        return self

    def transform(self, X):
        X = pd.DataFrame(X)

        return X.drop(
            columns=self.features_to_drop_,
            errors="ignore"
        )

In [20]:
corr_filter = CorrelationFilter(threshold=0.95)

X_train_corr = corr_filter.fit_transform(X_train_var)
X_valid_corr = corr_filter.transform(X_valid_var)

In [22]:
mi_filter = SelectKBest(
    score_func=mutual_info_regression,
    k=50
)

X_train_kbest = mi_filter.fit_transform(
    X_train_corr,
    y_train
)

X_valid_kbest = mi_filter.transform(
    X_valid_corr
)

In [24]:
selected_features = X_train_corr.columns[
    mi_filter.get_support()
]

print("Number of selected features:", len(selected_features))
print("Selected features:")
print(selected_features.tolist())

Number of selected features: 50
Selected features:
['20aa07010', '87ffda550', '963a49cdc', '935ca66a9', '861076e21', '26fc93eb7', '0572565c2', '66ace2992', '350473311', 'fb49e4212', '6619d81fc', '6eef030c1', 'fc99f9426', '1db387535', 'ce3d7595b', 'b43a7cfd5', '024c577b9', '2ec5b290f', '0ff32eb98', '166008929', '58e056e12', '241f0f867', '1931ccfdd', '58e2e02e6', '9fd594eec', 'fb0f5dbfe', '91f701ba2', 'ca2b906e8', '703885424', 'eeb9cd3aa', '324921c7b', '58232a6fb', '491b9ee45', 'c8d582dd2', 'd6bb78916', '70feb1494', 'adb64ff71', '9de83dc23', '62e59a501', '15ace8c9f', '5c6487af1', 'f190486d6', 'f74e8f13d', '77deffdf0', 'c5a231d81', 'e176a204a', '1702b5bf0', '190db8488', 'c47340d97', '23310aa6f']


In [29]:
X_train_kbest = pd.DataFrame(
    X_train_selected,
    columns=selected_features,
    index=X_train_corr.index
)

X_valid_kbest = pd.DataFrame(
    X_test_selected,
    columns=selected_features,
    index=X_test_corr.index
)

In [25]:
feature_scores = pd.DataFrame({
    "feature": X_train_corr.columns,
    "score": mi_filter.scores_,
})

feature_scores = feature_scores.sort_values(
    by="score",
    ascending=False
)

print(feature_scores)

        feature     score
629   6619d81fc  0.119900
3936  f190486d6  0.119792
612   fb49e4212  0.118441
524   26fc93eb7  0.115683
4243  190db8488  0.114009
...         ...       ...
327   8516b48f5  0.000000
329   f519ac616  0.000000
331   8f3807320  0.000000
335   0b26c77a9  0.000000
2325  9c404d218  0.000000

[4492 rows x 2 columns]


In [33]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_kbest,
    y_train
)

RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)

In [37]:
y_pred = model.predict(X_valid_kbest)

In [39]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    y_valid,
    y_pred
)

print("MAE:", mae)

MAE: 4814688.061580677


In [26]:


rf_selector = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train_kbest, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [30]:
feature_importance = pd.Series(
    rf_selector.feature_importances_,
    index=X_train_kbest.columns
).sort_values(ascending=False)

print(feature_importance)

f190486d6    0.108689
58e2e02e6    0.057425
eeb9cd3aa    0.052441
b43a7cfd5    0.039959
15ace8c9f    0.039673
20aa07010    0.031938
58232a6fb    0.030243
9fd594eec    0.029381
58e056e12    0.027517
1702b5bf0    0.027223
fb0f5dbfe    0.026960
6eef030c1    0.024865
5c6487af1    0.022746
d6bb78916    0.022100
f74e8f13d    0.021414
963a49cdc    0.020409
26fc93eb7    0.019002
491b9ee45    0.018535
66ace2992    0.017907
fb49e4212    0.017712
324921c7b    0.017357
23310aa6f    0.016934
70feb1494    0.016077
91f701ba2    0.015887
241f0f867    0.015612
190db8488    0.015539
024c577b9    0.015534
62e59a501    0.015446
2ec5b290f    0.014800
0572565c2    0.014551
1db387535    0.013733
703885424    0.013691
e176a204a    0.013226
c47340d97    0.012860
6619d81fc    0.012760
c5a231d81    0.012675
1931ccfdd    0.012268
0ff32eb98    0.012035
adb64ff71    0.011740
ce3d7595b    0.011210
87ffda550    0.010352
fc99f9426    0.009436
350473311    0.007024
c8d582dd2    0.006841
77deffdf0    0.005480
9de83dc23 

In [43]:
for k in [10, 20, 30, 40]:

    selected_features = feature_importance.head(k).index

    X_train_rf = X_train_kbest[selected_features]
    X_test_rf = X_valid_kbest[selected_features]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_rf, y_train)

    y_pred = model.predict(X_test_rf)

    mae = mean_absolute_error(y_valid, y_pred)

    print(f"{k} features -> MAE: {mae}")

10 features -> MAE: 5014802.146561943
20 features -> MAE: 5051085.238539285
30 features -> MAE: 4976468.068955123
40 features -> MAE: 4879923.139870016


# New Section